# Encontro 9 — Padrões de Colaboração entre Agentes e LangGraph

Este notebook apresenta padrões de colaboração entre agentes (Hierárquico, Debate, Cooperativo, Mercado, Contrato Net) e um exemplo prático de sistema multi-agente usando LangGraph, com um "agente pesquisador" que entrega seus resultados para um "agente escritor".

## Agenda
- Tipos de colaboração: Hierárquico, Debate, Cooperativo/Colaborativo, Mercado, Contrato Net
- Vantagens e desvantagens de cada padrão, com exemplos
- Implementação de fluxo multi-agente com LangGraph (pesquisador → escritor)
- Exemplos de Debate e Hierárquico usando LangGraph

## Padrão Hierárquico
**Como funciona:**
- Agentes organizados em níveis, com um líder/coordenador no topo e subordinados abaixo.
- O líder toma decisões globais ou delega tarefas; subordinados executam ações específicas.

**Vantagens:**
- Coordenação centralizada e tomada de decisão consistente.
- Reduz conflitos entre agentes.

**Desvantagens:**
- Ponto único de falha no líder.
- Menos flexível diante de mudanças rápidas.

**Exemplos práticos:**
- Robôs de limpeza em armazém: um supervisor decide áreas; subordinados limpam corredores específicos.
- IA generativa para relatórios: agente principal define roteiro; outros produzem seções específicas.

## Padrão de Debate
**Como funciona:**
- Agentes apresentam opiniões/argumentos sobre uma decisão.
- Um mecanismo de consenso (voto ou árbitro) escolhe a melhor opção.

**Vantagens:**
- Explora perspectivas diferentes.
- Melhora decisões em problemas complexos.

**Desvantagens:**
- Mais demorado; exige interação contínua.
- Pode gerar conflitos sem regras claras de arbitragem.

**Exemplos práticos:**
- Negociação de preços: cada agente propõe estratégia; um sistema de votação escolhe a melhor.
- Recomendação colaborativa: agentes debatem itens com base em métricas (preferência, popularidade, contexto).

## Padrão Cooperativo / Colaborativo
**Como funciona:**
- Trabalho distribuído; agentes compartilham informações e dividem tarefas sem líder fixo.

**Exemplos:**
- Drones mapeando uma área: cada drone cobre uma seção e compartilha dados.
- IA generativa: agentes criam partes do texto/análise, combinadas no resultado final.

## Padrão de Mercado (Market-Based)
**Como funciona:**
- Agentes competem por recursos/tarefas via leilões ou preços.

**Exemplo:**
- Alocação de tarefas em cloud computing: agentes ‘ofertam’ recursos necessários; sistema aloca baseado no leilão.

## Padrão de Contrato (Contract Net Protocol)
**Como funciona:**
- Um agente anuncia uma tarefa (‘contrato’); outros enviam propostas.
- O iniciador seleciona a melhor proposta; a tarefa é executada.

**Exemplo:**
- Robôs de entrega em armazém: um robô anuncia necessidade; outros se oferecem; o melhor é escolhido.

In [ ]:
# (Opcional) Instalar LangGraph caso não esteja presente
%pip install -q langgraph

In [ ]:
# Carregar LLM (mesmo padrão dos outros notebooks)
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
model_name = os.getenv("MODEL_NAME", "gemini-2.0-flash")
assert api_key, "GOOGLE_API_KEY ausente. Defina no .env."
llm = ChatGoogleGenerativeAI(model=model_name, google_api_key=api_key, streaming=False)
print("LLM pronto.")

## Exemplo prático: pesquisador → escritor com LangGraph
Fluxo simples com dois agentes: o pesquisador cria um resumo de pesquisa e o escritor monta um relatório baseado nesse resumo.
Para tornar o exemplo reproduzível sem credenciais, os agentes aqui são funções determinísticas (sem LLM).

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class PesquisaState(TypedDict):
    question: str
    research: str
    draft: str

prompt_researcher = ChatPromptTemplate.from_messages([
    ('system', 'Você é um pesquisador experiente. Produza um resumo estruturado em tópicos claros para a pergunta: {question}.'),
    ('human', 'Gere 4-6 tópicos com evidências e implicações.')
])
chain_researcher = prompt_researcher | llm | StrOutputParser()

def pesquisador(state: PesquisaState):
    text = chain_researcher.invoke({'question': state['question']})
    return {'research': text}

prompt_writer = ChatPromptTemplate.from_messages([
    ('system', 'Você é um escritor técnico. Construa um relatório sucinto com base na pesquisa e na pergunta.'),
    ('human', 'Pergunta: {question}\nPesquisa:\n{research}\n\nEscreva um relatório objetivo com seções: Introdução, Análise, Conclusões.')
])
chain_writer = prompt_writer | llm | StrOutputParser()

def escritor(state: PesquisaState):
    draft = chain_writer.invoke({'question': state['question'], 'research': state['research']})
    return {'draft': draft}

g = StateGraph(PesquisaState)
g.add_node('pesquisador', pesquisador)
g.add_node('escritor', escritor)
g.add_edge('pesquisador', 'escritor')
g.add_edge('escritor', END)
g.set_entry_point('pesquisador')
app = g.compile()

final = app.invoke({'question': 'Impacto da IA na educação'})
print(final['draft'])

### Observações
- O "estado" compartilha dados entre nós (agentes) no grafo.
- A transição pesquisador → escritor é definida por arestas; o término pelo `END`.
- Em produção, você pode substituir as funções por agentes LangChain (LLMs + prompts).

## Padrão de Debate com LangGraph
Dois agentes geram propostas e um árbitro escolhe a melhor. Aqui usamos uma regra simples (comprimento do texto) como critério.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class DebateState(TypedDict):
    prompt: str
    option_a: str
    option_b: str
    selected: str

prompt_a = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Agente A. Proponha uma solução focada em precisão para: {prompt}.'),
    ('human', 'Explique de forma objetiva.')
])
chain_a = prompt_a | llm | StrOutputParser()

def agente_a(state: DebateState):
    return {'option_a': chain_a.invoke({'prompt': state['prompt']})}

prompt_b = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Agente B. Proponha uma solução focada em velocidade para: {prompt}.'),
    ('human', 'Explique de forma objetiva.')
])
chain_b = prompt_b | llm | StrOutputParser()

def agente_b(state: DebateState):
    return {'option_b': chain_b.invoke({'prompt': state['prompt']})}

prompt_arbitro = ChatPromptTemplate.from_messages([
    ('system', 'Você é um árbitro. Escolha a melhor proposta entre A e B e retorne apenas o texto escolhido, sem comentários.'),
    ('human', 'Contexto: {prompt}\nProposta A:\n{option_a}\n\nProposta B:\n{option_b}\n\nEscolha a melhor.')
])
chain_arbitro = prompt_arbitro | llm | StrOutputParser()

def arbitro(state: DebateState):
    sel = chain_arbitro.invoke({'prompt': state['prompt'], 'option_a': state['option_a'], 'option_b': state['option_b']})
    return {'selected': sel}

g_db = StateGraph(DebateState)
g_db.add_node('A', agente_a)
g_db.add_node('B', agente_b)
g_db.add_node('Arbitro', arbitro)
g_db.add_edge('A', 'B')
g_db.add_edge('B', 'Arbitro')
g_db.add_edge('Arbitro', END)
g_db.set_entry_point('A')
app_db = g_db.compile()
out = app_db.invoke({'prompt': 'Escolher método de sumarização para relatório'})
print('Selecionado pelo árbitro:', out['selected'])

## Padrão Hierárquico com LangGraph
Um coordenador define o plano e delega a trabalhadores; um agregador combina os resultados.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class HierState(TypedDict):
    objetivo: str
    plano: str
    tarefa1: str
    tarefa2: str
    resultado: str

prompt_coord = ChatPromptTemplate.from_messages([
    ('system', 'Você é coordenador. Crie um plano de alto nível para: {objetivo}.'),
    ('human', 'Divida em duas tarefas claras.')
])
chain_coord = prompt_coord | llm | StrOutputParser()

def coordenador(state: HierState):
    plano = chain_coord.invoke({'objetivo': state['objetivo']})
    return {'plano': plano}

prompt_w1 = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Trabalhador 1. Execute a tarefa 1 baseada no plano.'),
    ('human', 'Plano:\n{plano}')
])
chain_w1 = prompt_w1 | llm | StrOutputParser()

def trabalhador1(state: HierState):
    t1 = chain_w1.invoke({'plano': state['plano']})
    return {'tarefa1': t1}

prompt_w2 = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Trabalhador 2. Execute a tarefa 2 baseada no plano.'),
    ('human', 'Plano:\n{plano}')
])
chain_w2 = prompt_w2 | llm | StrOutputParser()

def trabalhador2(state: HierState):
    t2 = chain_w2.invoke({'plano': state['plano']})
    return {'tarefa2': t2}

prompt_ag = ChatPromptTemplate.from_messages([
    ('system', 'Você é agregador. Combine os resultados das tarefas em um resumo final.'),
    ('human', 'Tarefa 1:\n{tarefa1}\n\nTarefa 2:\n{tarefa2}')
])
chain_ag = prompt_ag | llm | StrOutputParser()

def agregador(state: HierState):
    res = chain_ag.invoke({'tarefa1': state['tarefa1'], 'tarefa2': state['tarefa2']})
    return {'resultado': res}

g_h = StateGraph(HierState)
g_h.add_node('Coordenador', coordenador)
g_h.add_node('Worker1', trabalhador1)
g_h.add_node('Worker2', trabalhador2)
g_h.add_node('Agregador', agregador)
g_h.add_edge('Coordenador', 'Worker1')
g_h.add_edge('Worker1', 'Worker2')
g_h.add_edge('Worker2', 'Agregador')
g_h.add_edge('Agregador', END)
g_h.set_entry_point('Coordenador')
app_h = g_h.compile()
final_h = app_h.invoke({'objetivo': 'Gerar relatório de vendas trimestral'})
print(final_h['resultado'])

## Próximos passos
- Substituir funções determinísticas por agentes com LLMs (LangChain).
- Adicionar memória entre iterações e mecanismos de feedback.
- Implementar padrões Cooperativo e Mercado com LangGraph, incluindo coordenação de dados/recursos.

In [ ]:
# Cooperativo/Colaborativo com LLM: dois trabalhadores geram partes, combinador agrega via LLM
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class CoopState(TypedDict):
    objetivo: str
    part_a: str
    part_b: str
    combinado: str

prompt_a = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Trabalhador A. Gere a Parte A para o objetivo: {objetivo}.'),
    ('human', 'Seja objetivo.')
])
chain_a = prompt_a | llm | StrOutputParser()

prompt_b = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Trabalhador B. Gere a Parte B para o objetivo: {objetivo}.'),
    ('human', 'Seja objetivo.')
])
chain_b = prompt_b | llm | StrOutputParser()

prompt_comb = ChatPromptTemplate.from_messages([
    ('system', 'Você é combinador. Una as partes A e B em um resultado coerente.'),
    ('human', 'Parte A:\n{part_a}\n\nParte B:\n{part_b}')
])
chain_comb = prompt_comb | llm | StrOutputParser()

def worker_a(state: CoopState):
    return {'part_a': chain_a.invoke({'objetivo': state['objetivo']})}

def worker_b(state: CoopState):
    return {'part_b': chain_b.invoke({'objetivo': state['objetivo']})}

def combiner(state: CoopState):
    return {'combinado': chain_comb.invoke({'part_a': state['part_a'], 'part_b': state['part_b']})}

g_c = StateGraph(CoopState)
g_c.add_node('A', worker_a)
g_c.add_node('B', worker_b)
g_c.add_node('Combiner', combiner)
g_c.add_edge('A', 'B')
g_c.add_edge('B', 'Combiner')
g_c.add_edge('Combiner', END)
g_c.set_entry_point('A')
app_c = g_c.compile()
out_c = app_c.invoke({'objetivo': 'Gerar análise colaborativa'})
print(out_c['combinado'])

In [ ]:
# Mercado (Market-Based) com LLM: agentes fazem lances textuais, leiloeiro decide via LLM
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class MarketState(TypedDict):
    tarefa: str
    bid_a: str
    bid_b: str
    escolhido: str

prompt_bid_a = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Agente A. Faça um lance (custo/tempo) para a tarefa: {tarefa}.'),
    ('human', 'Inclua justificativa breve.')
])
chain_bid_a = prompt_bid_a | llm | StrOutputParser()

prompt_bid_b = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Agente B. Faça um lance (custo/tempo) para a tarefa: {tarefa}.'),
    ('human', 'Inclua justificativa breve.')
])
chain_bid_b = prompt_bid_b | llm | StrOutputParser()

prompt_auction = ChatPromptTemplate.from_messages([
    ('system', 'Você é leiloeiro. Escolha o melhor lance entre A e B com base em custo/tempo e qualidade. Retorne apenas "Agente A" ou "Agente B".'),
    ('human', 'Tarefa: {tarefa}\nLance A:\n{bid_a}\n\nLance B:\n{bid_b}\n\nQual escolher?')
])
chain_auction = prompt_auction | llm | StrOutputParser()

def bidder_a(state: MarketState):
    return {'bid_a': chain_bid_a.invoke({'tarefa': state['tarefa']})}

def bidder_b(state: MarketState):
    return {'bid_b': chain_bid_b.invoke({'tarefa': state['tarefa']})}

def auctioneer(state: MarketState):
    choice = chain_auction.invoke({'tarefa': state['tarefa'], 'bid_a': state['bid_a'], 'bid_b': state['bid_b']})
    return {'escolhido': choice}

g_m = StateGraph(MarketState)
g_m.add_node('A', bidder_a)
g_m.add_node('B', bidder_b)
g_m.add_node('Auctioneer', auctioneer)
g_m.add_edge('A', 'B')
g_m.add_edge('B', 'Auctioneer')
g_m.add_edge('Auctioneer', END)
g_m.set_entry_point('A')
app_m = g_m.compile()
res_m = app_m.invoke({'tarefa': 'Alocar recurso de CPU'})
print('Escolhido no leilão:', res_m['escolhido'])

In [ ]:
# Contrato (Contract Net Protocol) com LLM: propostas, seleção e execução via LLM
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class ContractState(TypedDict):
    tarefa: str
    proposta_a: str
    proposta_b: str
    selecionado: str
    resultado: str

prompt_ca = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Contratado A. Envie uma proposta para a tarefa: {tarefa} (custo/tempo/risco).'),
    ('human', 'Explique vantagens.')
])
chain_ca = prompt_ca | llm | StrOutputParser()

prompt_cb = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Contratado B. Envie uma proposta para a tarefa: {tarefa} (custo/tempo/risco).'),
    ('human', 'Explique vantagens.')
])
chain_cb = prompt_cb | llm | StrOutputParser()

prompt_sel = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Selecionador. Escolha a melhor proposta (A ou B) e retorne apenas o texto da proposta selecionada.'),
    ('human', 'Tarefa: {tarefa}\nProposta A:\n{proposta_a}\n\nProposta B:\n{proposta_b}\n\nSelecione a melhor.')
])
chain_sel = prompt_sel | llm | StrOutputParser()

prompt_exec = ChatPromptTemplate.from_messages([
    ('system', 'Você é o Executor. Descreva a execução baseada na proposta selecionada.'),
    ('human', 'Proposta selecionada:\n{selecionado}')
])
chain_exec = prompt_exec | llm | StrOutputParser()

def contractor_a(state: ContractState):
    return {'proposta_a': chain_ca.invoke({'tarefa': state['tarefa']})}

def contractor_b(state: ContractState):
    return {'proposta_b': chain_cb.invoke({'tarefa': state['tarefa']})}

def selector(state: ContractState):
    sel = chain_sel.invoke({'tarefa': state['tarefa'], 'proposta_a': state['proposta_a'], 'proposta_b': state['proposta_b']})
    return {'selecionado': sel}

def executor(state: ContractState):
    return {'resultado': chain_exec.invoke({'selecionado': state['selecionado']})}

g_cn = StateGraph(ContractState)
g_cn.add_node('ContractorA', contractor_a)
g_cn.add_node('ContractorB', contractor_b)
g_cn.add_node('Selector', selector)
g_cn.add_node('Executor', executor)
g_cn.add_edge('ContractorA', 'ContractorB')
g_cn.add_edge('ContractorB', 'Selector')
g_cn.add_edge('Selector', 'Executor')
g_cn.add_edge('Executor', END)
g_cn.set_entry_point('ContractorA')
app_cn = g_cn.compile()
out_cn = app_cn.invoke({'tarefa': 'Mover item pesado'})
print(out_cn['resultado'])